NOTE: The MWIS section of this notebook takes a long time to run (4000 models to run per SNP). It is recommended to switch to a VM with a high number of cores. 16 cores can complete this notebook in approximately 2-3 hours. HOWEVER, note that VMs with 32 cores or greater are unreliable and will cause mysterious errors in the sections parallelized with `parallel::mcMap()`.

# Setup
## Libraries and functions

In [ ]:
pkgs <- c('data.table','ggplot2','ggrepel','gtExtras','lme4','lme4qtl','lmerTest','lmtest','mediation','parallel','patchwork','geomtextpath')
if(!require(lme4qtl)) { remotes::install_github("variani/lme4qtl") } # Special install
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg,character.only=T)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

# Workaround so that lmerTest accepts lme4qtl model objects, so we can get lmerTest Satterthwaite p-values.
myLmer <- \(formula, data, relmat) {
  model <- relmatLmer(formula, data, relmat=relmat)
  lmerTest:::as_lmerModLT(model, as.function(model))
}

## Import data if not already present

In [ ]:
ws_bucket <- Sys.getenv('WORKSPACE_BUCKET')
system('mkdir -p data/derived/genomics')
system('mkdir -p data/derived/metabolomics/QCd/')
grab_file_if_not_extant <- \(f) if(!file.exists(f)) system(paste0('gcloud storage cp', ws_bucket,'/',f))
grab_file_if_not_extant('data/derived/analysis_df-mesa.csv')
grab_file_if_not_extant('data/derived/genomics/mesa_km.RData')
grab_file_if_not_extant('data/derived/metabolomics/QCd/merged_QCd-mesa.csv')
grab_file_if_not_extant('data/derived/metabolomics/met_info-mesa.csv')

## Prepare data

In [ ]:
data <- fread('data/derived/analysis_df-mesa.csv')
variants_of_interest <- fread(cmd='gcloud storage cat gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/variants_of_interest.csv')
cpaids <- variants_of_interest[cpaid %in% names(data), cpaid]

data <- fread('data/derived/analysis_df-mesa.csv')[
  ][, mesa_id   := as.factor(mesa_id)
  ][, exam      := as.factor(exam)
  ][, batch_cn  := as.factor(batch_cn)
  # Variants are numeric (0/1/2), but we temporarily set to character so they don't get scale()'d.
  ][, names(.SD) := lapply(.SD, as.character), .SDcols=cpaids
  ][, names(.SD) := lapply(.SD, scale       ), .SDcols=is.numeric
  ][, names(.SD) := lapply(.SD, as.numeric  ), .SDcols=cpaids
]

load('data/derived/genomics/mesa_km.RData') # Kinship (with ids remapped in script 01c)

met_nms <- names(fread('data/derived/metabolomics/QCd/merged_QCd-mesa.csv',nrows=0,drop=1))
covars <- c('site', 'sex', 'age', 'ses_score', 'drinks_per_week', 'smoking', 'ahei_score', paste0('gPC',1:9), 'race')
outcomes <- c('hdl_log','fg','hb','tg_log')
exposures <- c('bmi','mvpa_wins','sex','smoking')

calc_eff_n_metabolites <- \(met_nms) {
  met_nms <- met_nms[!is.na(met_nms)] |> unique()
  met_mtx <- data[ rowSums(is.na(data[,..met_nms]))==0, ..met_nms ] |> as.matrix(rownames=1) # Only samples having data for ALL metabolomics methods
  met_eigvals <- prcomp(met_mtx, scale=T, center=T)$sdev^2
  met_eff_n <- sum(met_eigvals)^2 / sum(met_eigvals^2)
}

# Test known associations
## Y~E associations
### Construct formulas
Model: `Y ~ E + covariates + (1|ID)`

In [ ]:
Y_E_models <-
CJ(Y=outcomes, E=exposures, model='Y~E')[
  ][, fmla := paste0('<Y> ~ <E>')                        # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := mapply(Y,E,fmla, FUN=\(Y,E,fmla) {         # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<E>',E,fmla) })
  ][, fmla := paste(fmla, '+', '(1|mesa_id)')            # Add random intercept.
]

### Display results
Only top 5 model terms are shown. E is highlighted.

In [ ]:
tbls <- Y_E_models[, .(Map(fmla,Y,E, f=\(fmla,Y,E) {
  myLmer(fmla, data, list(mesa_id=km)) |>
  summary() |> coef() |>
  as.data.table(keep.rownames='term') |>
  (\(dt) dt[order(`Pr(>|t|)`)])() |>
  head(5) |> gt(caption=paste(Y,'~',E)) |>
  gt_highlight_rows(rows=term==E) |>
  gt:::as.tags.gt_tbl()
}))]

In [ ]:
tbls$V1

## Y~G associations
### Construct formulas
Model: `Y ~ G + covariates + (1|ID)`

In [ ]:
Y_G_models <-
CJ(Y=outcomes, G=cpaids)[
  ][mapply(Y,G, FUN=\(Y,G) # Only keep Y&G combinations that appear in the variants_of_interest table.
      variants_of_interest[cpaid==G, grepl(Y,outcome)])
  ][, fmla := paste0('<Y> ~ <G>')                        # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := mapply(Y,G,fmla, FUN=\(Y,G,fmla) {         # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<G>',G,fmla) })
  ][, fmla := paste(fmla, '+', '(1|mesa_id)')            # Add random intercept.
  ][, term_pat := G                                      # Term whose β to extract.
]

### Run models

In [ ]:
options(mc.cores=4L)
i <- 0

t0 <- Sys.time()
Y_G_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]
Sys.time() - t0

### Display results

In [ ]:
tbls <- Y_G_models[
  ][variants_of_interest, on=.(G=cpaid)  # Merge in SNP metadata.
  ][!is.na(est)                          # Discard SNPs which weren't run.
  ][, .(Y,G,rsid,analysis,β_G=est,se,p)  # Keep only desired columns.
  ][, names(.SD) := lapply(.SD,signif,3) # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  gt(caption='Y~G associations') |>
  gt:::as.tags.gt_tbl()

In [ ]:
tbls

## E~G associations
### Construct formulas
Model: `E ~ G + covariates + (1|ID)`

In [ ]:
E_G_models <-
CJ(E=exposures, G=cpaids)[
  ][mapply(E,G, FUN=\(E,G) # Only keep E&G combinations that appear in the variants_of_interest table.
      variants_of_interest[cpaid==G, grepl(E,exposure)])
  ][E %in% names(data[,.SD,.SDcols=is.numeric])          # Outcome must be numeric.
  ][, fmla := paste0('<E> ~ <G>')                        # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := mapply(E,G,fmla, FUN=\(E,G,fmla) {         # Replace placeholders.
        fmla <- gsub('<E>',E,fmla)
        fmla <- gsub('<G>',G,fmla) })
  ][, fmla := paste(fmla, '+', '(1|mesa_id)')            # Add random intercept.
  ][, term_pat := G                                      # Term whose β to extract.
]

### Run models

In [ ]:
options(mc.cores=4L)
i <- 0
t0 <- Sys.time()
E_G_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]
Sys.time() - t0

### Display results

In [ ]:
E_G_models[
  ][variants_of_interest, on=.(G=cpaid)  # Merge in SNP metadata.
  ][!is.na(est)                          # Discard SNPs which weren't run.
  ][, .(E,G,rsid,analysis,β_G=est,se,p)  # Keep only desired columns.
  ][, names(.SD) := lapply(.SD,signif,3) # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  gt(caption='E~G associations') |>
  gt:::as.tags.gt_tbl()

## GxEs
### Construct formulas
Model: `Y ~ G*E + covariates + gPCs*E + (1|ID)`

In [ ]:
GxE_models <-
CJ(Y=outcomes, G=cpaids, E=exposures)[
  ][mapply(Y,E,G, FUN=\(Y,E,G) # Only keep Y&G and Y&E combinations that appear in the variants_of_interest table.
      variants_of_interest[cpaid==G, grepl(E,exposure) & grepl(Y,outcome)])
  ][E != 'smoking'                                              # >2 levels factor not worth the complication for just 1 SNP. Skip.
  ][, fmla := paste0('<Y> ~ <G>*<E>')                           # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+'))        # Add covars.
  ][, fmla := paste(fmla,'+',                                   # Add gPC*E covars.
        paste(collapse='+', paste0('gPC',1:9,'*<E>')))
  ][, fmla := mapply(Y,E,G,fmla, FUN=\(Y,E,G,fmla) {            # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<E>',E,fmla)
        fmla <- gsub('<G>',G,fmla) })
  ][, fmla := paste(fmla,'+ (1|mesa_id)')                       # Add random intercept.
  ][, term_pat := paste0(G,':',E,'|',E,':',G)                   # Term whose β to extract.
]

### Run models

In [ ]:
options(mc.cores=4L)
i <- 0
t0 <- Sys.time()
GxE_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]
Sys.time() - t0

### Display results

In [ ]:
GxE_models[
  ][variants_of_interest, on=.(G=cpaid)     # Merge in SNP metadata.
  ][!is.na(est)                             # Discard SNPs which weren't run.
  ][, .(Y,E,G,rsid,analysis,β_GxE=est,se,p) # Keep only desired columns.
  ][, names(.SD) := lapply(.SD,signif,3)    # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  gt(caption='GxEs') |>
  gt:::as.tags.gt_tbl()

## MWIS
Moving forward with only the following SNPs:

* CLASP1 variant from KEW 2024 (`chr2_121657822_C_T`)
* Strongest MESA-replicated GxBMI SNP from KEW 2022 (`chr16_56960982_G_A`)
* That one Phase 2 PA-Lipid SNP that's no longer significant (just b/c I want to see how much adding gPCs 6-9 changes the results) (`chr14_88305056_G_A`)

In [ ]:
main_snps <- c('chr2_121657822_C_T','chr16_56960982_G_A','chr14_88305056_G_A')

### Construct formulas
Model: `Y ~ G*M + G*E + covariates + gPCs*M + (1|ID)`

In [ ]:
GxMpGxE_models <-
CJ(Y=outcomes, E=exposures, G=main_snps, M=met_nms)[
  ][mapply(Y,E,G,M, FUN=\(Y,E,G,M) # Only keep Y&G and Y&E combinations that appear in the variants_of_interest table.
      variants_of_interest[cpaid==G, grepl(E,exposure) & grepl(Y,outcome)])
  ][, fmla := paste0('<Y> ~ <G>*<M>+<G>*<E>')            # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := paste(fmla,'+',                            # Add gPC*M covars.
        paste(collapse='+', paste0('gPC',1:9,'*<M>')))
  ][, fmla := paste(fmla, fcase(                         # Add metabolite batch covars.
          grepl(   '_C8_pos',M), '+ batch_cp',           #   (Depends on LC-MS method M came from.)
          grepl(  '_C18_neg',M), '+ batch_cn',
          grepl('_HILIC_pos',M), '+ batch_hp',
          grepl('_Amide_neg',M), '+ batch_an',
          default=''
      ))
  ][, fmla := mapply(Y,E,G,M,fmla, FUN=\(Y,E,G,M,fmla) {     # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<E>',E,fmla)
        fmla <- gsub('<G>',G,fmla)
        fmla <- gsub('<M>',M,fmla) })
  ][, fmla := paste(fmla,'+ (1|mesa_id)')                # Add random intercept.
  ][, term_pat := paste0(G,':',M,'|',M,':',G)            # Term whose β to extract.
]

### Run models

In [ ]:
options(mc.cores=4L)
i <- 0
t0 <- Sys.time()
GxMpGxE_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]
Sys.time() - t0

# All-exams LMs approximation (no longitudinal or kinship adjustment, but much faster)
#GxMpGxE_models[
#  ][, c('est','se','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
#        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
#        fmla <- gsub('+ (1|mesa_id)','',fmla, fixed=T)                       # Rm random intercept.
#        model <- lm(fmla,data)                                               # Run model.
#        coefs <- summary(model)$coefficients                                 # Extract coefs.
#        c(coefs[grepl(pat,rownames(coefs)),],
#          nrow(model$model)                 )
#      }))
#]

### Calculate p threshold

In [ ]:
n_eff_metabolites <- calc_eff_n_metabolites(met_nms)
print(n_eff_metabolites)
p_thresh_mwas <- 0.05/n_eff_metabolites

# Alternate threshold: only counting metaboilites available for replication.
# Requires running the 04* scripts first.
# Does not actually make much difference though (number of effective metabolites: 68.7 -> 64.3)
# met_info_aligned <- fread('data/derived/metabolomics/met_info-aligned.csv')
# met_nms_aligned <- met_nms[met_nms %in% met_info_aligned$unique_met_id]
# n_eff_metabolites_aligned <- calc_eff_n_metabolites(met_nms_aligned)
# p_thresh_mwas_aligned <- 0.05/n_eff_metabolites_aligned

### Display results

In [ ]:
signif_GxM_tbl <- fread('data/derived/metabolomics/met_info-mesa.csv')[
  ][GxMpGxE_models, on=.(unique_met_id=M)   # Merge in met metadata.
  ][variants_of_interest, on=.(G=cpaid)     # Merge in SNP metadata.
  ][!is.na(est)                             # Discard SNPs which weren't run.
  ][, .(Y,E,G,rsid,analysis,β_GxM=est,se,p, # Keep only desired columns.
        M=unique_met_id, Metabolite)
  ][, names(.SD) := lapply(.SD,signif,3)    # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  #][p < p_thresh_mwas_aligned
  ][p < p_thresh_mwas
  ][order(p)
  ][analysis!='PA-Lipid' # Don't actually want this SNP, was just curious to see if +gPC6:9 significantly changed the results (not really).
]

signif_mets <- signif_GxM_tbl[analysis!='PA-Lipid']$M
fwrite(signif_GxM_tbl, 'mesa_signif_GxMs.csv')

signif_GxM_tbl |>
  dplyr::group_by(G,rsid,analysis) |>
  gt(caption='GxMs') |>
  gt:::as.tags.gt_tbl()

fwrite(GxMpGxE_models,'GxM_results.csv')

## Sensitivity analyses
### M~E
#### Construct formulas
Model: `M ~ E + covariates + (1|ID)`

In [ ]:
M_E_models <-
CJ(M=signif_mets, E=exposures)[
  ][mapply(M,E, FUN=\(M,E) # Only keep M~E combos relevant to the signif GxMs.
      any(M==signif_GxM_tbl$M & E==signif_GxM_tbl$E))
  ][, fmla := paste0('<M> ~ <E>')                        # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := mapply(M,E,fmla, FUN=\(M,E,fmla) {         # Replace placeholders.
        fmla <- gsub('<M>',M,fmla)
        fmla <- gsub('<E>',E,fmla) })
  ][, fmla := paste(fmla, '+', '(1|mesa_id)')            # Add random intercept.
  ][, term_pat := E                                      # Term whose β to extract.
]

#### Run models

In [ ]:
options(mc.cores=4L)
i <- 0
M_E_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]

#### Display results

In [ ]:
fread('data/derived/metabolomics/met_info-mesa.csv')[
  ][M_E_models, on=.(unique_met_id=M)    # Merge in met metadata.
  ][, .(M=unique_met_id,E,β_E=est,se,p)  # Keep only desired columns.
  ][, names(.SD) := lapply(.SD,signif,3) # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  dplyr::group_by(E) |>
  gt(caption='M~E associations') |>
  gt:::as.tags.gt_tbl()

### Extra GxCovars covariates
#### Construct formulas
Model: `Y ~ G*M + G*E + covariates + gPCs*M + G*covariates (1|ID)`

In [ ]:
GxMpGxEpGxC_models <-
CJ(Y=outcomes, E=exposures, G=main_snps, M=signif_mets)[
  ][mapply(Y,E,G,M, FUN=\(Y,E,G,M)# Only keep signif GxMs from the regular analysis.
      any(Y==signif_GxM_tbl$Y & E==signif_GxM_tbl$E & G==signif_GxM_tbl$G & M==signif_GxM_tbl$M))
  ][, fmla := paste0('<Y> ~ <G>*<M>+<G>*<E>')            # Construct formulas.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := paste(fmla,'+',                            # Add G*covars.
        paste(paste0('<G>*',covars),collapse=' + '))     #
  ][, fmla := gsub('\\+ <G>\\*gPC[0-9]','',fmla)         #   Rm <G>*gPC terms though.
  ][, fmla := paste(fmla,'+',                            # Add gPC*M covars.
        paste(collapse='+', paste0('gPC',1:9,'*<M>')))
  ][, fmla := paste(fmla, fcase(                         # Add metabolite batch covars.
          grepl(   '_C8_pos',M), '+ <G>*batch_cp',       #   (Depends on LC-MS method M came from.)
          grepl(  '_C18_neg',M), '+ <G>*batch_cn',
          grepl('_HILIC_pos',M), '+ <G>*batch_hp',
          grepl('_Amide_neg',M), '+ <G>*batch_an',
          default=''
      ))
  ][, fmla := mapply(Y,E,G,M,fmla, FUN=\(Y,E,G,M,fmla) { # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<E>',E,fmla)
        fmla <- gsub('<G>',G,fmla)
        fmla <- gsub('<M>',M,fmla) })
  ][, fmla := paste(fmla,'+ (1|mesa_id)')                # Add random intercept.
  ][, term_pat := paste0(G,':',M,'|',M,':',G)            # Term whose β to extract.
]

### Run models

In [ ]:
options(mc.cores=4L)
i <- 0
GxMpGxEpGxC_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(mesa_id=km))                          # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]

### Display results

In [ ]:
fread('data/derived/metabolomics/met_info-mesa.csv')[
  ][GxMpGxEpGxC_models, on=.(unique_met_id=M) # Merge in met metadata.
  ][variants_of_interest, on=.(G=cpaid)       # Merge in SNP metadata.
  ][!is.na(est)                               # Discard SNPs which weren't run.
  ][, .(Y,G,rsid,analysis,β_GxM=est,se,p,     # Keep only desired columns.
        M=unique_met_id, Metabolite)
  ][, names(.SD) := lapply(.SD,signif,3)      # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  dplyr::group_by(G,rsid,analysis) |>
  gt(caption='GxMs (sensitivity analysis, with +GxCovars)') |>
  gt:::as.tags.gt_tbl()